In [1]:
import os
import numpy as np
import pandas as pd

from sklearn.metrics import mean_squared_error, mean_absolute_error

In [2]:
print("Current working directory:", os.getcwd())
print("Files here:", os.listdir("."))

Current working directory: /home/sagemaker-user
Files here: ['.bashrc', '.sagemaker_sql_editor_api_cache', '.local', '.ipython', '.aws', '.npm', '.jupyter', '.ipynb_checkpoints', '.cache', '.config', 'netflix-data.zip', 'netflix_data', 'dataprep', 'training', 'data_exploration_netflix.ipynb', 'data_prep_netflix.ipynb', 'Untitled-Copy1.ipynb', 'factorization_model.ipynb', 'train.csv', 'validation.csv', 'model_evaluations.ipynb', 'test.csv', 'data_prep_netflix_final.ipynb', 'model_evaluations_final.ipynb', 'user-default-efs', '.virtual_documents']


In [3]:
import boto3
import os

bucket = "aaron-netflix-ads508-665606011923-us-east-1-an"
s3 = boto3.client("s3")

print("Current working directory:", os.getcwd())
print("train path:", os.path.abspath("train.csv"))
print("validation path:", os.path.abspath("validation.csv"))
print("test path:", os.path.abspath("test.csv"))

response = s3.list_objects_v2(Bucket=bucket, Prefix="netflix_clean_csv_v2/")
for obj in response.get("Contents", []):
    print(obj["Key"], obj["LastModified"], obj["Size"])

Current working directory: /home/sagemaker-user
train path: /home/sagemaker-user/train.csv
validation path: /home/sagemaker-user/validation.csv
test path: /home/sagemaker-user/test.csv
netflix_clean_csv_v2/test.csv 2026-04-10 15:19:51+00:00 1784464
netflix_clean_csv_v2/train.csv 2026-04-10 15:19:50+00:00 13447455
netflix_clean_csv_v2/validation.csv 2026-04-10 15:19:51+00:00 1694975


In [4]:
train_df = pd.read_csv("train.csv")
validation_df = pd.read_csv("validation.csv")
test_df = pd.read_csv("test.csv")

print("Train:", train_df.shape)
print("Validation:", validation_df.shape)
print("Test:", test_df.shape)

print("\ntrain_df columns:")
print(train_df.columns.tolist())

print("\nvalidation_df columns:")
print(validation_df.columns.tolist())

print("\ntest_df columns:")
print(test_df.columns.tolist())

train_df.head()

Train: (320000, 6)
Validation: (40000, 6)
Test: (40000, 6)

train_df columns:
['user_id', 'movie_id', 'rating', 'year', 'title', 'release_year']

validation_df columns:
['user_id', 'movie_id', 'rating', 'year', 'title', 'release_year']

test_df columns:
['user_id', 'movie_id', 'rating', 'year', 'title', 'release_year']


,user_id,movie_id,rating,year,title,release_year
0,471064,4503,3,1999,Grace of My Heart,1996
1,1723352,9235,4,1999,G.I. Jane,1997
2,996461,13383,4,2000,If These Walls Could Talk 2,2000
3,829693,18,5,2000,Immortal Beloved,1994
4,1139570,16,3,2000,Screamers,1996


In [5]:
print("Train years:")
print(train_df["year"].value_counts().sort_index())

print("\nValidation years:")
print(validation_df["year"].value_counts().sort_index())

print("\nTest years:")
print(test_df["year"].value_counts().sort_index())

Train years:
year
1999         2
2000      3414
2001      8152
2002     17583
2003     48620
2004    126598
2005    115631
Name: count, dtype: int64

Validation years:
year
2005    40000
Name: count, dtype: int64

Test years:
year
2005    40000
Name: count, dtype: int64


In [15]:
train_model = train_df[["user_id", "movie_id", "rating"]].copy()
validation_model = validation_df[["user_id", "movie_id", "rating"]].copy()
test_model = test_df[["user_id", "movie_id", "rating"]].copy()

print(train_model.shape)
print(validation_model.shape)
print(test_model.shape)

(320000, 3)
(40000, 3)
(40000, 3)


In [6]:
def evaluate_predictions(y_true, y_pred, model_name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)

    return pd.DataFrame({
        "Model": [model_name],
        "RMSE": [rmse],
        "MAE": [mae]
    })

In [7]:
global_mean = train_df["rating"].mean()

user_bias = train_df.groupby("user_id")["rating"].mean() - global_mean
movie_bias = train_df.groupby("movie_id")["rating"].mean() - global_mean

print("Global mean:", global_mean)

Global mean: 3.55436875


In [8]:
test_bias = test_df.copy()

test_bias["user_bias"] = test_bias["user_id"].map(user_bias).fillna(0)
test_bias["movie_bias"] = test_bias["movie_id"].map(movie_bias).fillna(0)

test_bias["prediction"] = global_mean + test_bias["user_bias"] + test_bias["movie_bias"]
test_bias["prediction"] = test_bias["prediction"].clip(1, 5)

bias_results = evaluate_predictions(
    test_bias["rating"],
    test_bias["prediction"],
    "User-Movie Bias Model"
)

bias_results

,Model,RMSE,MAE
0,User-Movie Bias Model,1.316153,1.015448


In [9]:
user_item_train = train_df.pivot_table(
    index="user_id",
    columns="movie_id",
    values="rating"
)

print(user_item_train.shape)
user_item_train.head()

(145289, 99)


movie_id,1,2,3,4,5,6,7,8,9,10,...,13380,13381,13382,13383,13384,13385,13386,13387,13388,13389
user_id,,,,,,,,,,,,,,,,,,,,,
6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
59,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,4.0,NaN,NaN,NaN,NaN,NaN,NaN
79,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
97,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,4.0,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
user_item_filled = user_item_train.apply(
    lambda row: row.fillna(row.mean()),
    axis=1
)

# if a user row is entirely NaN, fall back to the global mean
global_mean = train_df["rating"].mean()
user_item_filled = user_item_filled.fillna(global_mean)

print(user_item_filled.shape)
user_item_filled.head()

(145289, 99)


movie_id,1,2,3,4,5,6,7,8,9,10,...,13380,13381,13382,13383,13384,13385,13386,13387,13388,13389
user_id,,,,,,,,,,,,,,,,,,,,,
6,3.00,3.00,3.00,3.00,3.00,3.00,3.00,3.00,3.00,3.00,...,3.00,3.00,3.00,3.00,3.00,3.00,3.00,3.00,3.00,3.00
7,4.25,4.25,4.25,4.25,4.25,4.25,4.25,4.25,4.25,4.25,...,4.25,4.25,4.25,4.25,4.25,4.25,4.25,4.25,4.25,4.25
59,3.00,3.00,3.00,3.00,3.00,3.00,3.00,3.00,3.00,3.00,...,3.00,3.00,3.00,4.00,3.00,3.00,3.00,3.00,3.00,3.00
79,3.00,3.00,3.00,3.00,3.00,3.00,3.00,3.00,3.00,3.00,...,3.00,3.00,3.00,3.00,3.00,3.00,3.00,3.00,3.00,3.00
97,3.00,3.00,3.00,3.00,3.00,3.00,3.00,3.00,3.00,3.00,...,3.00,3.00,3.00,4.00,3.00,3.00,3.00,3.00,3.00,3.00


In [11]:
from sklearn.decomposition import NMF

nmf_model = NMF(
    n_components=20,
    init="random",
    random_state=42,
    max_iter=200
)

W = nmf_model.fit_transform(user_item_filled)
H = nmf_model.components_

print("W shape:", W.shape)
print("H shape:", H.shape)

/opt/conda/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1728: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(


W shape: (145289, 20)
H shape: (20, 99)


In [12]:
reconstructed = np.dot(W, H)

nmf_predictions = pd.DataFrame(
    reconstructed,
    index=user_item_filled.index,
    columns=user_item_filled.columns
)

nmf_predictions.head()

movie_id,1,2,3,4,5,6,7,8,9,10,...,13380,13381,13382,13383,13384,13385,13386,13387,13388,13389
user_id,,,,,,,,,,,,,,,,,,,,,
6,3.000350,2.997422,3.040741,3.004536,2.989872,3.020787,3.076767,2.960385,2.965192,3.032301,...,2.975943,3.078106,3.055851,2.947422,3.029132,2.992781,3.064242,3.058384,3.117232,3.090998
7,4.256839,4.223976,4.240152,4.239773,4.259579,4.274936,4.215238,4.308565,4.286086,4.239462,...,4.278661,4.239425,4.236151,4.446346,4.234741,4.272874,4.221182,4.208632,4.236822,4.344264
59,2.991121,3.017712,3.007164,3.007220,2.998828,2.982644,2.998457,2.966616,2.986522,3.000645,...,2.991502,2.990255,2.986283,3.269109,2.998390,2.985626,3.007790,3.017923,3.049937,3.235353
79,2.999247,3.000367,2.999985,2.999667,3.000484,2.998438,2.999154,3.000972,3.000042,2.998664,...,3.000549,2.998916,2.992305,3.006923,2.998846,3.001030,2.999306,2.998478,3.000432,3.004254
97,2.994907,3.015391,3.005690,3.009932,2.997322,2.987484,2.998759,2.960742,2.989586,3.004778,...,2.988662,2.993896,3.039580,3.237393,3.012454,2.979877,3.008681,3.021511,3.052969,3.183376


In [13]:
test_nmf = test_df.copy()

global_mean = train_df["rating"].mean()

def predict_nmf(user_id, movie_id):
    if user_id in nmf_predictions.index and movie_id in nmf_predictions.columns:
        return nmf_predictions.loc[user_id, movie_id]
    return global_mean

test_nmf["prediction"] = test_nmf.apply(
    lambda row: predict_nmf(row["user_id"], row["movie_id"]),
    axis=1
)

test_nmf["prediction"] = test_nmf["prediction"].clip(1, 5)

nmf_results = evaluate_predictions(
    test_nmf["rating"],
    test_nmf["prediction"],
    "NMF / Matrix Factorization"
)

nmf_results

,Model,RMSE,MAE
0,NMF / Matrix Factorization,1.243401,0.984588


In [14]:
evaluation_results = pd.concat([bias_results, nmf_results], ignore_index=True)
evaluation_results

,Model,RMSE,MAE
0,User-Movie Bias Model,1.316153,1.015448
1,NMF / Matrix Factorization,1.243401,0.984588


In [15]:
best_baseline = evaluation_results.sort_values("RMSE").iloc[0]["Model"]

print("Baseline comparison complete.")
print("Best baseline by RMSE:", best_baseline)

Baseline comparison complete.
Best baseline by RMSE: NMF / Matrix Factorization


In [16]:
import sagemaker
from sagemaker import image_uris, model_uris, script_uris
from sagemaker.estimator import Estimator
from sagemaker.deserializers import JSONDeserializer
from sagemaker.serializers import CSVSerializer
from sagemaker.predictor import Predictor

sess = sagemaker.Session()
role = sagemaker.get_execution_role()
region = sess.boto_region_name

print("Region:", region)
print("Role:", role)

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
Region: us-east-1
Role: arn:aws:iam::665606011923:role/LabRole


In [17]:
bucket = "aaron-netflix-ads508-665606011923-us-east-1-an"

fm_model_artifact = (
    f"s3://{bucket}/models/"
    "factorization-machines-2026-04-10-09-52-10-629/output/model.tar.gz"
)

print(fm_model_artifact)

s3://aaron-netflix-ads508-665606011923-us-east-1-an/models/factorization-machines-2026-04-10-09-52-10-629/output/model.tar.gz


In [18]:
fm_inference_image = sagemaker.image_uris.retrieve(
    framework="factorization-machines",
    region=region
)

print(fm_inference_image)

382416733822.dkr.ecr.us-east-1.amazonaws.com/factorization-machines:1


In [19]:
from sagemaker.model import Model

fm_model = Model(
    image_uri=fm_inference_image,
    model_data=fm_model_artifact,
    role=role,
    sagemaker_session=sess
)

In [20]:
fm_predictor = fm_model.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    serializer=CSVSerializer(),
    deserializer=JSONDeserializer()
)

print(type(fm_predictor))
print(fm_predictor)

------------!<class 'NoneType'>
None


In [21]:
#finding endpoint name

sm_client = sess.sagemaker_client

response = sm_client.list_endpoints(
    SortBy="CreationTime",
    SortOrder="Descending"
)

for ep in response["Endpoints"][:10]:
    print(ep["EndpointName"], ep["EndpointStatus"])

factorization-machines-2026-04-10-20-13-25-640 InService
factorization-machines-2026-04-10-08-16-19-721 InService
factorization-machines-2026-04-10-04-36-21-891 InService


In [22]:
X_test = test_df[["user_id", "movie_id", "release_year"]].copy()
y_test = test_df["rating"].copy()

print(X_test.shape)
X_test.head()

(40000, 3)


,user_id,movie_id,release_year
0,1780095,13384,1979
1,287034,13384,1979
2,821222,13384,1979
3,2194798,13384,1979
4,1379931,13384,1979


In [23]:
import io
import numpy as np
import sagemaker.amazon.common as smac

X_test_np = X_test.astype("float32").to_numpy()

small_batch_np = X_test_np[:100]

buf = io.BytesIO()
smac.write_numpy_to_dense_tensor(buf, small_batch_np)
payload = buf.getvalue()

print(type(payload), len(payload))

<class 'bytes'> 3600


In [24]:
from sagemaker.predictor import Predictor
from sagemaker.serializers import IdentitySerializer

endpoint_name = "factorization-machines-2026-04-10-20-13-25-640"

fm_predictor = Predictor(
    endpoint_name=endpoint_name,
    sagemaker_session=sess,
    serializer=IdentitySerializer(content_type="application/x-recordio-protobuf"),
    deserializer=JSONDeserializer()
)

print(type(fm_predictor))
print(fm_predictor.endpoint_name)

<class 'sagemaker.base_predictor.Predictor'>
factorization-machines-2026-04-10-20-13-25-640


In [25]:
small_result = fm_predictor.predict(payload)
small_result

{'predictions': [{'score': -34470473728.0},
  {'score': -5924368896.0},
  {'score': -16137695232.0},
  {'score': -42401902592.0},
  {'score': -26817966080.0},
  {'score': -43408535552.0},
  {'score': -39893708800.0},
  {'score': -19202158592.0},
  {'score': -44058652672.0},
  {'score': -17141182464.0},
  {'score': -39528804352.0},
  {'score': -27633758208.0},
  {'score': -6944829952.0},
  {'score': -42859081728.0},
  {'score': -41462378496.0},
  {'score': -12006043648.0},
  {'score': -31807090688.0},
  {'score': -46428434432.0},
  {'score': -43140100096.0},
  {'score': -25202016256.0},
  {'score': -58652639232.0},
  {'score': -53720137728.0},
  {'score': -10823155712.0},
  {'score': -51417464832.0},
  {'score': -4814946304.0},
  {'score': -61827727360.0},
  {'score': -61966139392.0},
  {'score': -15628517376.0},
  {'score': -17699454976.0},
  {'score': -49613914112.0},
  {'score': -11541168128.0},
  {'score': -39560167424.0},
  {'score': -21508407296.0},
  {'score': -60405858304.0},
  

In [26]:
small_preds = [p["score"] for p in small_result["predictions"]]
print(len(small_preds))
print(small_preds[:5])

100
[-34470473728.0, -5924368896.0, -16137695232.0, -42401902592.0, -26817966080.0]


In [27]:
batch_size = 500
fm_preds = []

for start in range(0, len(X_test_np), batch_size):
    end = start + batch_size
    batch_np = X_test_np[start:end]

    buf = io.BytesIO()
    smac.write_numpy_to_dense_tensor(buf, batch_np)
    batch_payload = buf.getvalue()

    batch_result = fm_predictor.predict(batch_payload)
    batch_scores = [p["score"] for p in batch_result["predictions"]]
    fm_preds.extend(batch_scores)

print("Total predictions:", len(fm_preds))
print("First 5 predictions:", fm_preds[:5])

Total predictions: 40000
First 5 predictions: [-34470473728.0, -5924368896.0, -16137695232.0, -42401902592.0, -26817966080.0]


In [28]:
test_fm = test_df.copy()
test_fm["prediction"] = np.clip(fm_preds, 1, 5)

fm_results = evaluate_predictions(
    test_fm["rating"],
    test_fm["prediction"],
    "Factorization Machines"
)

fm_results

,Model,RMSE,MAE
0,Factorization Machines,2.691148,2.427575


In [29]:
print("train_df columns:")
print(train_df.columns.tolist())

print("\nvalidation_df columns:")
print(validation_df.columns.tolist())

print("\ntest_df columns:")
print(test_df.columns.tolist())

train_df columns:
['user_id', 'movie_id', 'rating', 'year', 'title', 'release_year']

validation_df columns:
['user_id', 'movie_id', 'rating', 'year', 'title', 'release_year']

test_df columns:
['user_id', 'movie_id', 'rating', 'year', 'title', 'release_year']


In [30]:
from xgboost import XGBRegressor

xgb_features = ["user_id", "movie_id", "year", "release_year"]

X_train_xgb = train_df[xgb_features].copy()
y_train_xgb = train_df["rating"].copy()

X_val_xgb = validation_df[xgb_features].copy()
y_val_xgb = validation_df["rating"].copy()

X_test_xgb = test_df[xgb_features].copy()
y_test_xgb = test_df["rating"].copy()

print("Using XGBoost features:", xgb_features)
print("Train shape:", X_train_xgb.shape)
print("Validation shape:", X_val_xgb.shape)
print("Test shape:", X_test_xgb.shape)

Using XGBoost features: ['user_id', 'movie_id', 'year', 'release_year']
Train shape: (320000, 4)
Validation shape: (40000, 4)
Test shape: (40000, 4)


In [31]:
xgb_model = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

xgb_model.fit(
    X_train_xgb,
    y_train_xgb,
    eval_set=[(X_val_xgb, y_val_xgb)],
    verbose=False
)

xgb_preds = xgb_model.predict(X_test_xgb)
xgb_preds = np.clip(xgb_preds, 1, 5)

xgb_results = evaluate_predictions(
    y_test_xgb,
    xgb_preds,
    "XGBoost Regressor"
)

xgb_results

,Model,RMSE,MAE
0,XGBoost Regressor,1.267698,1.00066


In [32]:
final_results = pd.concat(
    [bias_results, nmf_results, xgb_results, fm_results],
    ignore_index=True
)

final_results = final_results.sort_values("RMSE").reset_index(drop=True)
final_results

,Model,RMSE,MAE
0,NMF / Matrix Factorization,1.243401,0.984588
1,XGBoost Regressor,1.267698,1.000660
2,User-Movie Bias Model,1.316153,1.015448
3,Factorization Machines,2.691148,2.427575


In [33]:
best_model = final_results.sort_values("RMSE").iloc[0]["Model"]

print("Final model evaluation comparison complete.")
print("Best model by RMSE:", best_model)
print("NMF/Matrix Factorization performed best overall on the test set.")
print("Factorization Machines produced the weakest results.")

Final model evaluation comparison complete.
Best model by RMSE: NMF / Matrix Factorization
NMF/Matrix Factorization performed best overall on the test set.
Factorization Machines produced the weakest results.


In [34]:
# deleting most recent endpoint

try:
    sm_client.delete_endpoint(EndpointName=endpoint_name)
    print(f"Deleted endpoint: {endpoint_name}")
except Exception as e:
    print(f"Could not delete endpoint {endpoint_name}: {e}")

Deleted endpoint: factorization-machines-2026-04-10-20-13-25-640
